In [1]:
!nvidia-smi

Mon May  4 01:44:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from google.colab import files
uploaded = files.upload()


Saving test.csv to test.csv
Saving train.csv to train.csv


In [6]:
from datasets import load_dataset
ds = load_dataset("eth-nlped/mathdial")
print("Splits:", list(ds.keys()))
train = ds["train"].to_pandas()
print("Train shape:", train.shape)
print("Columns:", list(train.columns))
train.head(2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/2262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/599 [00:00<?, ? examples/s]

Splits: ['train', 'test']
Train shape: (2262, 11)
Columns: ['qid', 'scenario', 'question', 'ground_truth', 'student_incorrect_solution', 'student_profile', 'teacher_described_confusion', 'self-correctness', 'self-typical-confusion', 'self-typical-interactions', 'conversation']


,qid,scenario,question,ground_truth,student_incorrect_solution,student_profile,teacher_described_confusion,self-correctness,self-typical-confusion,self-typical-interactions,conversation
0,5000012,1,Nancy is filling an aquarium for her fish. She...,First calculate the volume of the aquarium by ...,The aquarium has a volume of 4 x 6 x 3 = 72 cu...,Steven is a 7th grade student. He has difficul...,He added a step after completing the problem.,Yes,3.0,3.0,"Teacher: (probing)Steven, If you had 4 of some..."
1,5000084,2,John is very unfit and decides to work up to d...,He needs to do 15*3=45 progressions\nThat will...,"To get to 15 reps, John will take 15 - 1 = 14 ...",Stephanie is a 7th grade student. She has diff...,She became fixated on a wrong calculation and ...,No,2.0,2.0,"Teacher: (probing)Stephanie, How many days wil..."


In [7]:
"""
Build preference pairs from MathDial conversations.

Logic:
  1. Parse each conversation string into (Teacher, Student) turn list.
  2. For each Teacher turn, build a context = preceding student turn
     (the student message that the teacher is responding to).
  3. Label each Teacher turn with its conversation-level outcome:
       Yes                          -> "good" (preferred)
       No                           -> "bad"  (dispreferred)
       Yes, but I had to reveal...  -> "bad"  (dispreferred)
  4. Sample pairs by matching contexts of similar length & taking
     one good + one bad teacher turn. The model learns: given the
     same student context, which teacher turn was better?
"""

import pandas as pd
import numpy as np
import re

OUTCOME_GOOD = {"Yes"}
OUTCOME_BAD = {"No", "Yes, but I had to reveal the answer"}


def parse_conversation(conv: str):
    """Split a conversation string into a list of (role, text) tuples."""
    if not isinstance(conv, str):
        return []
    # Conversations use "Teacher:" / "Student:" markers, sometimes
    # inside parentheses there's a tutor-move tag like "(probing)".
    # We strip those tags but keep the actual message.
    parts = re.split(r"(Teacher:|Student:)", conv)
    parts = [p.strip() for p in parts if p.strip()]
    turns = []
    i = 0
    while i < len(parts) - 1:
        if parts[i] in ("Teacher:", "Student:"):
            role = parts[i].rstrip(":")
            text = parts[i + 1]
            # Strip a leading move tag like "(probing)" or "(focus)"
            text = re.sub(r"^\([^)]+\)\s*", "", text)
            turns.append((role, text.strip()))
            i += 2
        else:
            i += 1
    return turns


def extract_teacher_turns_with_context(row):
    """Return a list of dicts: {context, teacher_turn, label} per row."""
    turns = parse_conversation(row["conversation"])
    outcome = row.get("self-correctness", "")
    if outcome in OUTCOME_GOOD:
        label = "good"
    elif outcome in OUTCOME_BAD:
        label = "bad"
    else:
        return []  # skip rows with missing/unknown outcome

    out = []
    prev_student = ""
    for role, text in turns:
        if role == "Student":
            prev_student = text
        elif role == "Teacher":
            if len(text) < 5 or len(prev_student) < 3:
                continue  # too-short turns are noise
            out.append({
                "qid": row.get("qid", ""),
                "context": prev_student,
                "teacher_turn": text,
                "label": label,
            })
    return out


# Build the turn-level table
all_turns = []
for _, row in train.iterrows():
    all_turns.extend(extract_teacher_turns_with_context(row))

turns_df = pd.DataFrame(all_turns)
print(f"Extracted {len(turns_df)} teacher turns")
print(turns_df["label"].value_counts())
turns_df.head(3)

Extracted 11627 teacher turns
label
good    7536
bad     4091
Name: count, dtype: int64


,qid,context,teacher_turn,label
0,5000084,"It will take one day to do one step, since Joh...",So how many days will it take to do 15 wall pu...,bad
1,5000084,"It will take 15 days to do 15 wall push-ups, s...",How many days to get to 15 high elevation pus-...,bad
2,5000084,It will take 29 days to get to 15 high elevati...,"Re-calculate, you jusy said iy took 15 days to...",bad


In [8]:
"""
Build pairs: (context, good_turn, bad_turn).
We pair within randomly matched contexts of comparable length to keep
the comparison fair. This is the standard pairwise-preference setup.
"""

from sklearn.model_selection import train_test_split

good = turns_df[turns_df["label"] == "good"].reset_index(drop=True)
bad = turns_df[turns_df["label"] == "bad"].reset_index(drop=True)

print(f"Good turns: {len(good)}, Bad turns: {len(bad)}")

# Build pairs: for each bad turn, sample a good turn whose context
# has roughly similar length. This avoids trivial "long vs short" cues.
rng = np.random.default_rng(42)
good["ctx_len"] = good["context"].str.len()
bad["ctx_len"] = bad["context"].str.len()

pairs = []
for _, b in bad.iterrows():
    # Find good turns with context length within +/- 30% of bad's context length
    lo = max(1, int(b["ctx_len"] * 0.7))
    hi = max(lo + 1, int(b["ctx_len"] * 1.3))
    candidates = good[(good["ctx_len"] >= lo) & (good["ctx_len"] <= hi)]
    if len(candidates) == 0:
        candidates = good
    g = candidates.iloc[int(rng.integers(len(candidates)))]
    pairs.append({
        "context": b["context"],            # use bad's context for symmetry
        "good_turn": g["teacher_turn"],
        "bad_turn": b["teacher_turn"],
    })

pairs_df = pd.DataFrame(pairs)
print(f"Built {len(pairs_df)} preference pairs")

# 80/20 train/val split for the scorer
pairs_train, pairs_val = train_test_split(pairs_df, test_size=0.2, random_state=42)
print(f"Pairs train: {len(pairs_train)}, val: {len(pairs_val)}")
pairs_train.head(2)

Good turns: 7536, Bad turns: 4091
Built 4091 preference pairs
Pairs train: 3272, val: 819


,context,good_turn,bad_turn
2465,"Yes, that's right. I only applied the voucher ...",Are you sure? The calculation is right but yo...,"How much money did the entree cost, according ..."
2823,He charged $50/10 = $5 per action figure.|EOM|,Whatever way works best for you to solve a pro...,He earned $100 from selling the action figures...


In [9]:
"""
Fine-tune DeBERTa-v3-base as a cross-encoder pairwise scorer.
Architecture: standard sequence classifier head outputs a scalar logit.
Loss: margin-ranking loss between (good, bad) pair scores.
"""

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
MARGIN = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PairwiseDataset(Dataset):
    def __init__(self, pairs_df, tokenizer, max_len=MAX_LEN):
        self.df = pairs_df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def _encode(self, ctx, turn):
        return self.tok(
            ctx, turn,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        good = self._encode(r["context"], r["good_turn"])
        bad = self._encode(r["context"], r["bad_turn"])
        return {
            "good_input_ids": good["input_ids"].squeeze(0),
            "good_attention_mask": good["attention_mask"].squeeze(0),
            "bad_input_ids": bad["input_ids"].squeeze(0),
            "bad_attention_mask": bad["attention_mask"].squeeze(0),
        }


class PreferenceScorer(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use CLS-style pooling: first-token hidden state
        cls = out.last_hidden_state[:, 0]
        score = self.head(cls).squeeze(-1)
        return score


train_ds = PairwiseDataset(pairs_train, tokenizer)
val_ds = PairwiseDataset(pairs_val, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = PreferenceScorer().to(device)
optim = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optim, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)


def pairwise_acc(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            for k in batch:
                batch[k] = batch[k].to(device)
            sg = model(batch["good_input_ids"], batch["good_attention_mask"])
            sb = model(batch["bad_input_ids"], batch["bad_attention_mask"])
            correct += (sg > sb).sum().item()
            total += sg.size(0)
    return correct / total


print(f"Starting training: {EPOCHS} epochs, {len(train_loader)} steps/epoch")
for epoch in range(EPOCHS):
    model.train()
    losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        for k in batch:
            batch[k] = batch[k].to(device)
        sg = model(batch["good_input_ids"], batch["good_attention_mask"])
        sb = model(batch["bad_input_ids"], batch["bad_attention_mask"])
        # Margin ranking: good should score higher than bad by MARGIN
        loss = torch.clamp(MARGIN - (sg - sb), min=0).mean()

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        scheduler.step()
        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")

    val_acc = pairwise_acc(model, val_loader)
    print(f"Epoch {epoch+1}: avg train loss={np.mean(losses):.4f}, val pairwise acc={val_acc:.4f}")

Device: cuda


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

Starting training: 2 epochs, 409 steps/epoch


Epoch 1:   0%|          | 0/409 [00:00<?, ?it/s]

Epoch 1: avg train loss=0.1625, val pairwise acc=0.9585


Epoch 2:   0%|          | 0/409 [00:00<?, ?it/s]

Epoch 2: avg train loss=0.0363, val pairwise acc=0.9719


In [10]:
import os
SAVE_DIR = "/content/pps_deberta"
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), f"{SAVE_DIR}/pps_state.pt")
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to", SAVE_DIR)

# Zip it for download
!cd /content && zip -r pps_deberta.zip pps_deberta/ -q
print("Zipped.")

from google.colab import files
files.download("/content/pps_deberta.zip")

Saved to /content/pps_deberta
Zipped.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>